# CICIoMT2024 — Publication Full Run Notebook (Patched)

نسخة مصححة بعد خطأ MCC overflow، مع حماية إضافية للذاكرة والحفظ.


In [1]:

# وظيفة الخلية: ربط Google Drive بشكل صريح، لأن بيانات CICIoMT2024 محفوظة في Drive.
from google.colab import drive

# force_remount يقلل مشاكل mount القديمة أو المنقطعة قبل قراءة ملفات CSV الكبيرة.
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [2]:

# وظيفة الخلية: ضبط إعدادات التشغيل النهائي الكامل قبل النشر، مع قراءة البيانات من Drive ثم نسخها محلياً قبل التحليل.
import os
from datetime import datetime
from pathlib import Path

RANDOM_STATE = 42
TRAIN_RATIO = 0.60
VALID_RATIO = 0.20
TEST_RATIO = 0.20
assert abs(TRAIN_RATIO + VALID_RATIO + TEST_RATIO - 1.0) < 1e-9

# شغّل المقارنة الأساسية بين dedup_on و dedup_off.
RUN_MAIN = True
RUN_DEDUP_ON = True
RUN_DEDUP_OFF = True

# OCSVM هو الأبطأ؛ اتركه مفعلاً للنتائج النهائية، وأوقفه فقط لاختبار الدفتر بسرعة.
RUN_OCSVM_MAIN = True

# تحليل البذور المتعددة مفعّل هنا لأن هذه نسخة التشغيل النهائي الواحد قبل النشر.
RUN_REPEATED_SEEDS = True
RUN_OCSVM_REPEATED = True
REPEATED_SEEDS = [0, 1, 2, 3, 4]

# تحليل العتبات مطلوب للورقة ولا يضيف تدريباً جديداً.
RUN_THRESHOLD_SENSITIVITY = True
THRESHOLD_PERCENTILES = [90, 95, 97.5, 99]

# attack-train لا يدخل التدريب، لكنه يُحمّل هنا فقط لتدقيق التداخل وإثبات عدم استخدامه في الملاءمة.
LOAD_ATTACK_TRAIN_FOR_AUDIT = True

# حفظ التنبؤات الكاملة مفعّل لأن هذه نسخة تشغيل نهائي واحدة للأرشفة والتدقيق.
SAVE_LARGE_PREDICTION_CSV = True
SAVE_BENIGN_SENSITIVITY_PREDICTIONS = True

# أهم تعديل لحل Errno 107: لا نقرأ CSV مباشرة من Google Drive أثناء التحليل.
# أولاً ننسخ مجلدات CSV إلى /content ثم نقرأ محلياً؛ هذا أسرع وأكثر ثباتاً مع الملفات الكبيرة.
STAGE_DATA_TO_LOCAL = True
LOCAL_STAGING_ROOT = '/content/iomt_staged_data'
LOCAL_DATASET_ROOT = os.path.join(LOCAL_STAGING_ROOT, 'WiFI_and_MQTT')

# تُحفظ النتائج محلياً أولاً لتجنب انقطاع Google Drive أثناء كتابة الملفات الكبيرة.
# في نهاية الدفتر يتم ضغط مجلد النتائج ونسخه إلى Drive.
WRITE_OUTPUTS_LOCAL_FIRST = True
WRITE_RESULTS_ZIP_TO_DRIVE = True
LOCAL_EXPERIMENT_BASE = '/content/iomt_publication_runs'

# إذا كان مسار WiFI_and_MQTT مختلفاً، ضع المسار الكامل هنا، مثال:
# DATASET_ROOT_MANUAL = '/content/drive/MyDrive/YourProject/DataSet/CICIoMT2024/WiFI_and_MQTT'
DATASET_ROOT_MANUAL = ''

# تحديد جذر المشروع داخل Google Drive لاستخدامه فقط للعثور على البيانات ونسخ ملف النتائج النهائي.
PROJECT_ROOT_CANDIDATES = [
    '/content/drive/MyDrive/IoMT_Project',
    '/content/drive/MyDrive',
    os.getcwd(),
]
PROJECT_ROOT = next((p for p in PROJECT_ROOT_CANDIDATES if os.path.exists(p)), os.getcwd())

# نبحث عن مجلد WiFI_and_MQTT في المسارات المتوقعة دون بحث عميق مكلف في Drive.
def first_existing_path(paths):
    for p in paths:
        if p and os.path.isdir(p):
            return p
    return paths[0] if paths else ''

DATASET_ROOT_CANDIDATES = [
    DATASET_ROOT_MANUAL,
    os.path.join(PROJECT_ROOT, 'DataSet', 'CICIoMT2024', 'WiFI_and_MQTT'),
    os.path.join(PROJECT_ROOT, 'DataSet', 'CICIoMT2024', 'WiFi_and_MQTT'),
    os.path.join(PROJECT_ROOT, 'datasets', 'CICIoMT2024', 'WiFI_and_MQTT'),
    os.path.join(PROJECT_ROOT, 'datasets', 'CICIoMT2024', 'WiFi_and_MQTT'),
]
DATASET_ROOT = first_existing_path(DATASET_ROOT_CANDIDATES)

STRICT_RUN_TAG = datetime.now().strftime('publication_full_%Y%m%d_%H%M%S')

DRIVE_EXPERIMENT_PARENT = os.path.join(
    PROJECT_ROOT,
    'Experiments',
    'CICIoMT2024_NormalOnly_DuplicateAware_publication_full',
)

EXPERIMENT_ROOT = os.path.join(
    LOCAL_EXPERIMENT_BASE if WRITE_OUTPUTS_LOCAL_FIRST else DRIVE_EXPERIMENT_PARENT,
    STRICT_RUN_TAG,
)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATASET_ROOT =', DATASET_ROOT)
print('STAGE_DATA_TO_LOCAL =', STAGE_DATA_TO_LOCAL)
print('LOCAL_DATASET_ROOT =', LOCAL_DATASET_ROOT)
print('EXPERIMENT_ROOT =', EXPERIMENT_ROOT)
print('DRIVE_EXPERIMENT_PARENT =', DRIVE_EXPERIMENT_PARENT)


PROJECT_ROOT = /content/drive/MyDrive/IoMT_Project
DATASET_ROOT = /content/drive/MyDrive/IoMT_Project/DataSet/CICIoMT2024/WiFI_and_MQTT
STAGE_DATA_TO_LOCAL = True
LOCAL_DATASET_ROOT = /content/iomt_staged_data/WiFI_and_MQTT
EXPERIMENT_ROOT = /content/iomt_publication_runs/publication_full_20260506_013718
DRIVE_EXPERIMENT_PARENT = /content/drive/MyDrive/IoMT_Project/Experiments/CICIoMT2024_NormalOnly_DuplicateAware_publication_full


In [3]:
# وظيفة الخلية: استيراد المكتبات اللازمة فقط دون تثبيت تلقائي أو مسارات تشغيل محلي.
import gc
import glob
import hashlib
import json
import math
import time
import shutil
import subprocess
import re


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

SENTINEL = -999999.0
META_COLS = ['__source_file', '__source_group', '__aux_label']


In [4]:
# وظيفة الخلية: إنشاء مجلدات الإخراج وتوثيق إعدادات التشغيل قبل بدء التجربة.
CONFIG_DIR = os.path.join(EXPERIMENT_ROOT, 'config')
SPLITS_DIR = os.path.join(EXPERIMENT_ROOT, 'splits')
METRICS_DIR = os.path.join(EXPERIMENT_ROOT, 'metrics')
PREDICTIONS_DIR = os.path.join(EXPERIMENT_ROOT, 'predictions')
FIGURES_DIR = os.path.join(EXPERIMENT_ROOT, 'figures')
AUDITS_DIR = os.path.join(EXPERIMENT_ROOT, 'audits')
SUMMARIES_DIR = os.path.join(EXPERIMENT_ROOT, 'summaries')
PUBLICATION_DIR = os.path.join(EXPERIMENT_ROOT, 'publication_checks')

for d in [CONFIG_DIR, SPLITS_DIR, METRICS_DIR, PREDICTIONS_DIR, FIGURES_DIR, AUDITS_DIR, SUMMARIES_DIR, PUBLICATION_DIR]:
    os.makedirs(d, exist_ok=True)

RUN_CONFIG = {
    'project_root': PROJECT_ROOT,
    'dataset_root': DATASET_ROOT,
    'experiment_root': EXPERIMENT_ROOT,
    'random_state': RANDOM_STATE,
    'train_ratio': TRAIN_RATIO,
    'valid_ratio': VALID_RATIO,
    'test_ratio': TEST_RATIO,
    'run_main': RUN_MAIN,
    'run_dedup_on': RUN_DEDUP_ON,
    'run_dedup_off': RUN_DEDUP_OFF,
    'run_ocsvm_main': RUN_OCSVM_MAIN,
    'run_repeated_seeds': RUN_REPEATED_SEEDS,
    'run_ocsvm_repeated': RUN_OCSVM_REPEATED,
    'repeated_seeds': REPEATED_SEEDS,
    'run_threshold_sensitivity': RUN_THRESHOLD_SENSITIVITY,
    'threshold_percentiles': THRESHOLD_PERCENTILES,
    'load_attack_train_for_audit': LOAD_ATTACK_TRAIN_FOR_AUDIT,
    'save_large_prediction_csv': SAVE_LARGE_PREDICTION_CSV,
}


In [5]:
# وظيفة الخلية: تعريف دوال الحفظ والتحويل الآمن للأنواع الرقمية والجداول الكبيرة.
def json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, (pd.Timestamp,)):
        return str(o)
    return str(o)


def save_json(obj, path):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=json_default)


def downcast_numeric(df):
    out = df.copy()
    for c in out.select_dtypes(include=['float64']).columns:
        out[c] = pd.to_numeric(out[c], downcast='float')
    for c in out.select_dtypes(include=['int64']).columns:
        out[c] = pd.to_numeric(out[c], downcast='integer')
    return out


def optimize_prediction_table(df):
    # يقلل استهلاك الذاكرة قبل حفظ جداول التنبؤات الكبيرة دون تغيير القيم.
    out = df.copy()
    category_cols = ['__source_file', '__source_group', '__aux_label', 'model', 'setting', 'treatment']
    for c in category_cols:
        if c in out.columns:
            out[c] = out[c].astype('category')
    for c in ['final_label', 'prediction']:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], downcast='integer')
    for c in ['anomaly_score']:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], downcast='float')
    return out


def safe_to_parquet(df, path):
    # يحفظ Parquet إن كانت المكتبة متاحة؛ وإذا فشل يحفظ Pickle حتى لا يتوقف الدفتر بعد تشغيل طويل.
    try:
        df.to_parquet(path, index=False)
        return path
    except Exception as e:
        fallback = os.path.splitext(path)[0] + '.pkl'
        df.to_pickle(fallback)
        print(f'تحذير: فشل حفظ Parquet في {path}. تم الحفظ كـ Pickle بدلاً منه: {fallback}. السبب: {type(e).__name__}: {e}')
        return fallback


def safe_token(value):
    # يحول أسماء الإعدادات والملفات إلى أسماء آمنة للملفات الناتجة.
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value)).strip('_')


def check_drive_mount():
    # يتحقق من أن Google Drive ما زال متصلاً قبل النسخ منه أو إليه.
    try:
        test_path = '/content/drive/MyDrive'
        return os.path.isdir(test_path) and os.access(test_path, os.R_OK)
    except OSError:
        return False


def copy_tree_with_rsync_or_python(src_dir, dst_dir):
    # ينسخ مجلد CSV من Drive إلى /content. rsync أسرع، وshutil بديل إذا لم يتوفر.
    os.makedirs(dst_dir, exist_ok=True)
    if not os.path.isdir(src_dir):
        raise FileNotFoundError(f'Missing source directory for staging: {src_dir}')

    rsync_cmd = ['rsync', '-a', '--ignore-existing', src_dir.rstrip('/') + '/', dst_dir.rstrip('/') + '/']
    try:
        result = subprocess.run(rsync_cmd, check=False, capture_output=True, text=True)
        if result.returncode == 0:
            return {'method': 'rsync', 'src': src_dir, 'dst': dst_dir}
        print('تحذير: rsync فشل، سيتم استخدام Python copy. stderr:', result.stderr[-500:])
    except Exception as e:
        print('تحذير: rsync غير متاح أو فشل، سيتم استخدام Python copy. السبب:', repr(e))

    copied = 0
    for root, _, files in os.walk(src_dir):
        rel = os.path.relpath(root, src_dir)
        out_root = os.path.join(dst_dir, rel) if rel != '.' else dst_dir
        os.makedirs(out_root, exist_ok=True)
        for fn in files:
            src_fp = os.path.join(root, fn)
            dst_fp = os.path.join(out_root, fn)
            if os.path.exists(dst_fp) and os.path.getsize(dst_fp) == os.path.getsize(src_fp):
                continue
            shutil.copy2(src_fp, dst_fp)
            copied += 1
    return {'method': 'python_copy', 'src': src_dir, 'dst': dst_dir, 'copied_files': copied}


def stage_dataset_to_local(dataset_root):
    # ينسخ مجلدات CSV المطلوبة محلياً لتجنب خطأ Transport endpoint عند القراءة المباشرة من Google Drive.
    if not STAGE_DATA_TO_LOCAL:
        return dataset_root, []

    if not os.path.isdir(dataset_root):
        raise FileNotFoundError(
            'DATASET_ROOT غير موجود. عدّل DATASET_ROOT_MANUAL إلى مسار WiFI_and_MQTT الصحيح: ' + str(dataset_root)
        )

    staging_records = []
    pairs = [
        ('profiling/CSV', os.path.join(dataset_root, 'profiling', 'CSV'), os.path.join(LOCAL_DATASET_ROOT, 'profiling', 'CSV')),
        ('attacks/CSV/test', os.path.join(dataset_root, 'attacks', 'CSV', 'test'), os.path.join(LOCAL_DATASET_ROOT, 'attacks', 'CSV', 'test')),
    ]
    if LOAD_ATTACK_TRAIN_FOR_AUDIT:
        pairs.append(('attacks/CSV/train', os.path.join(dataset_root, 'attacks', 'CSV', 'train'), os.path.join(LOCAL_DATASET_ROOT, 'attacks', 'CSV', 'train')))

    print('بدء نسخ CSV من Drive إلى التخزين المحلي في Colab...')
    for label, src_dir, dst_dir in pairs:
        if not os.path.isdir(src_dir):
            if label == 'attacks/CSV/train':
                print('تحذير: مجلد train غير موجود وسيتخطاه التدقيق:', src_dir)
                continue
            raise FileNotFoundError(f'Missing required source directory: {src_dir}')
        record = copy_tree_with_rsync_or_python(src_dir, dst_dir)
        record['label'] = label
        staging_records.append(record)
        print(f'تم تجهيز {label}: {dst_dir}')

    return LOCAL_DATASET_ROOT, staging_records


In [6]:

# وظيفة الخلية: تجهيز مصدر قراءة CSV ثم تحميل الملفات مع الاحتفاظ باسم الملف والمجموعة كمعلومات تفسيرية.
def infer_aux_label(filename, source_group):
    if source_group == 'profiling':
        return 'normal'
    return 'normal' if 'benign' in filename.lower() else 'attack'


def load_csv_group(csv_dir, source_group):
    files = sorted(glob.glob(os.path.join(csv_dir, '*.csv')))
    if not files:
        return pd.DataFrame(), []
    frames = []
    file_audit = []
    for i, fp in enumerate(files, start=1):
        name = os.path.basename(fp)
        try:
            df = pd.read_csv(fp)
        except OSError as e:
            raise OSError(
                f'فشل قراءة {fp}. إذا ظهر Errno 107 فهذا يعني أن Google Drive انقطع أثناء القراءة. '
                f'استخدم النسخة التي تنسخ البيانات محلياً أو أعد تشغيل runtime ثم شغل الدفتر من البداية. الأصل: {e}'
            )
        df = downcast_numeric(df)
        df['__source_file'] = name
        df['__source_group'] = source_group
        df['__aux_label'] = infer_aux_label(name, source_group)
        frames.append(df)
        file_audit.append({'file': name, 'rows': len(df), 'cols': len(df.columns)})
        if i % 10 == 0 or i == len(files):
            print(f'Loaded {source_group}: {i}/{len(files)} files')
    return pd.concat(frames, ignore_index=True), file_audit


def resolve_data_source():
    active_dataset_root, staging_records = stage_dataset_to_local(DATASET_ROOT)

    profiling_dir = os.path.join(active_dataset_root, 'profiling', 'CSV')
    attack_test_dir = os.path.join(active_dataset_root, 'attacks', 'CSV', 'test')
    attack_train_dir = os.path.join(active_dataset_root, 'attacks', 'CSV', 'train')

    required = [profiling_dir, attack_test_dir]
    missing_required = [d for d in required if not (os.path.isdir(d) and glob.glob(os.path.join(d, '*.csv')))]
    if missing_required:
        raise FileNotFoundError(f'Missing required CSV folders after staging/checking: {missing_required}')

    profiling_df, profiling_audit = load_csv_group(profiling_dir, 'profiling')
    attack_test_df, attack_test_audit = load_csv_group(attack_test_dir, 'attack_test')

    attack_train_df = None
    attack_train_audit = []
    if LOAD_ATTACK_TRAIN_FOR_AUDIT:
        if os.path.isdir(attack_train_dir) and glob.glob(os.path.join(attack_train_dir, '*.csv')):
            attack_train_df, attack_train_audit = load_csv_group(attack_train_dir, 'attack_train')
        else:
            print('تحذير: attack-train غير موجود؛ سيتم تجاهل تدقيقه.')

    info = {
        'mode': 'staged_local_csv' if STAGE_DATA_TO_LOCAL else 'raw_csv_only',
        'original_dataset_root': DATASET_ROOT,
        'active_dataset_root': active_dataset_root,
        'staging_records': staging_records,
        'profiling_dir': profiling_dir,
        'attack_test_dir': attack_test_dir,
        'attack_train_dir': attack_train_dir if LOAD_ATTACK_TRAIN_FOR_AUDIT else None,
        'profiling_files': profiling_audit,
        'attack_test_files': attack_test_audit,
        'attack_train_files': attack_train_audit,
    }
    return profiling_df, attack_test_df, attack_train_df, info


In [7]:
# وظيفة الخلية: بناء مخطط الخصائص من profiling فقط ثم محاذاة باقي المجموعات عليه لمنع إدخال خصائص هجومية خاصة.
def profiling_feature_schema(profiling_df):
    numeric_cols = profiling_df.select_dtypes(include=[np.number]).columns.tolist()
    schema = [c for c in numeric_cols if not c.startswith('__') and c not in ['__binary_label']]
    return sorted(schema)


def apply_schema(df, schema, meta_cols=META_COLS):
    if df is None:
        return None, {'missing_from_schema': [], 'extra_numeric_ignored': [], 'available': False}
    out = df.copy()
    numeric_cols = [c for c in out.select_dtypes(include=[np.number]).columns.tolist() if not c.startswith('__') and c not in ['__binary_label']]
    missing = [c for c in schema if c not in out.columns]
    extra = [c for c in numeric_cols if c not in schema]
    for c in meta_cols:
        if c not in out.columns:
            out[c] = np.nan
    passthrough_meta = [c for c in out.columns if c.startswith('__') and c not in meta_cols]
    aligned = out.reindex(columns=schema + meta_cols + passthrough_meta, fill_value=np.nan)
    for c in schema:
        aligned[c] = pd.to_numeric(aligned[c], errors='coerce')
    return aligned, {'missing_from_schema': missing, 'extra_numeric_ignored': extra, 'available': True}


In [8]:
# وظيفة الخلية: تعريف دوال الهاش والتكرار والتداخل لضمان split مستقل وخالٍ من التسرب المباشر.
def row_hashes(df, feature_cols):
    stable = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(SENTINEL)
    return pd.util.hash_pandas_object(stable, index=False).astype(str)


def duplicate_summary(df, name):
    if df is None:
        return {'pool': name, 'rows': None, 'unique_rows': None, 'duplicate_rows': None, 'duplicate_ratio': None, 'available': False}
    total = len(df)
    unique = int(df['__row_hash'].nunique()) if total else 0
    duplicates = int(total - unique)
    ratio = float(duplicates / total) if total else 0.0
    return {'pool': name, 'rows': int(total), 'unique_rows': unique, 'duplicate_rows': duplicates, 'duplicate_ratio': ratio, 'available': True}


def overlap_count(df_a, df_b):
    if df_a is None or df_b is None:
        return None
    return int(len(set(df_a['__row_hash']).intersection(set(df_b['__row_hash']))))


def assert_disjoint(df_a, df_b, name_a, name_b):
    overlap = overlap_count(df_a, df_b)
    assert overlap == 0, f'Leakage detected بين {name_a} و {name_b}: {overlap} overlapping hashes.'


In [9]:
# وظيفة الخلية: تعريف دوال التقسيم والمعالجة؛ كل المعالجة تُدرّب على normal_train فقط.
def split_by_hash(df, seed):
    unique_hashes = df['__row_hash'].drop_duplicates().tolist()
    train_h, temp_h = train_test_split(unique_hashes, test_size=(VALID_RATIO + TEST_RATIO), random_state=seed, shuffle=True)
    valid_h, test_h = train_test_split(temp_h, test_size=(TEST_RATIO / (VALID_RATIO + TEST_RATIO)), random_state=seed, shuffle=True)
    train_df = df[df['__row_hash'].isin(train_h)].copy().reset_index(drop=True)
    valid_df = df[df['__row_hash'].isin(valid_h)].copy().reset_index(drop=True)
    test_df = df[df['__row_hash'].isin(test_h)].copy().reset_index(drop=True)
    return train_df, valid_df, test_df


def fit_preprocessors(train_df, feature_cols):
    imputer = SimpleImputer(strategy='median')
    scaler = StandardScaler()
    X_train = train_df[feature_cols].replace([np.inf, -np.inf], np.nan)
    X_train_imp = imputer.fit_transform(X_train)
    X_train_scaled = scaler.fit_transform(X_train_imp)
    return imputer, scaler, X_train_scaled


def transform_features(df, feature_cols, imputer, scaler):
    X = df[feature_cols].replace([np.inf, -np.inf], np.nan)
    X_imp = imputer.transform(X)
    X_scaled = scaler.transform(X_imp)
    return X_scaled


def make_final_test(normal_test_df, attack_test_clean):
    normal_part = normal_test_df.copy()
    normal_part['final_label'] = 0
    anomaly_side = attack_test_clean.copy()
    anomaly_side['final_label'] = 1
    final_df = pd.concat([normal_part, anomaly_side], ignore_index=True)
    assert set(final_df['final_label'].unique()) <= {0, 1}
    return final_df


In [10]:
# وظيفة الخلية: تعريف النماذج والمقاييس بما فيها MCC مع توحيد اتجاه anomaly score.
def build_models(run_ocsvm=True, seed=RANDOM_STATE):
    models = {
        'isolation_forest': IsolationForest(n_estimators=200, contamination='auto', random_state=seed, n_jobs=-1),
        'lof': LocalOutlierFactor(n_neighbors=35, novelty=True),
    }
    if run_ocsvm:
        models['ocsvm'] = OneClassSVM(kernel='rbf', gamma='scale', nu=0.05)
    return models


def safe_div(num, den):
    return float(num) / float(den) if den else np.nan


def confusion_to_metrics(tn, fp, fn, tp):
    # يحوّل قيم confusion matrix إلى int عادي لتجنب overflow في MCC مع الجداول الكبيرة.
    tn, fp, fn, tp = int(tn), int(fp), int(fn), int(tp)
    total = tn + fp + fn + tp

    accuracy = safe_div(tn + tp, total)
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    specificity = safe_div(tn, tn + fp)
    fpr = safe_div(fp, tn + fp)
    prevalence = safe_div(tp + fn, total)

    if np.isfinite(precision) and np.isfinite(recall) and (precision + recall) > 0:
        f1 = 2.0 * precision * recall / (precision + recall)
    else:
        f1 = np.nan

    if np.isfinite(recall) and np.isfinite(specificity):
        balanced_accuracy = (recall + specificity) / 2.0
    else:
        balanced_accuracy = np.nan

    mcc_denom = float(tp + fp) * float(tp + fn) * float(tn + fp) * float(tn + fn)
    if mcc_denom > 0:
        mcc = ((float(tp) * float(tn)) - (float(fp) * float(fn))) / math.sqrt(mcc_denom)
    else:
        mcc = np.nan

    return {
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
        'mcc': float(mcc),
        'specificity': float(specificity),
        'fpr': float(fpr),
        'balanced_accuracy': float(balanced_accuracy),
        'pr_auc_baseline': float(prevalence),
    }


def evaluate_scores(y_true, scores, threshold):
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    preds = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()

    unique_y = np.unique(y_true)
    if len(unique_y) == 2:
        roc_auc = float(roc_auc_score(y_true, scores))
        pr_auc = float(average_precision_score(y_true, scores))
    else:
        roc_auc = np.nan
        pr_auc = np.nan

    metrics = {
        'roc_auc': float(roc_auc),
        'pr_auc': float(pr_auc),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'threshold': float(threshold),
    }
    metrics.update(confusion_to_metrics(tn, fp, fn, tp))
    return metrics, preds


def score_summary(scores, threshold):
    x = np.asarray(scores, dtype=float)
    pct_above = float((x >= threshold).mean() * 100.0) if len(x) else np.nan
    return {
        'count': int(len(x)),
        'mean_score': float(np.mean(x)) if len(x) else np.nan,
        'median_score': float(np.median(x)) if len(x) else np.nan,
        'std_score': float(np.std(x)) if len(x) else np.nan,
        'min_score': float(np.min(x)) if len(x) else np.nan,
        'q1_score': float(np.quantile(x, 0.25)) if len(x) else np.nan,
        'q3_score': float(np.quantile(x, 0.75)) if len(x) else np.nan,
        'max_score': float(np.max(x)) if len(x) else np.nan,
        'threshold': float(threshold),
        'pct_above_threshold': pct_above,
    }


In [11]:
# وظيفة الخلية: رسم المنحنيات ومصفوفة الالتباس باستخدام matplotlib فقط لتقليل الاعتماد على مكتبات إضافية.
def plot_confusion_matrix(cm, title, out_path):
    fig, ax = plt.subplots(figsize=(5.2, 4.8), dpi=320)
    im = ax.imshow(cm)
    ax.set_xticks([0, 1], labels=['Pred Normal', 'Pred Anomaly-side'])
    ax.set_yticks([0, 1], labels=['True Profiling Normal', 'True Anomaly-side'])
    ax.set_title(title)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f'{cm[i, j]:,}', ha='center', va='center')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(out_path, dpi=320, bbox_inches='tight')
    plt.close(fig)


def plot_score_boxplot(score_groups, threshold, title, out_path):
    labels = []
    values = []
    for label, arr in score_groups:
        arr = np.asarray(arr, dtype=float)
        if len(arr) == 0:
            continue
        take = min(len(arr), 50000)
        if len(arr) > take:
            idx = np.random.default_rng(RANDOM_STATE).choice(len(arr), size=take, replace=False)
            arr = arr[idx]
        labels.append(label)
        values.append(arr)
    if not values:
        return
    fig, ax = plt.subplots(figsize=(7.6, 5.6), dpi=320)
    ax.boxplot(values, labels=labels, showfliers=False)
    ax.axhline(threshold, linestyle='--', linewidth=1.5, label=f'threshold={threshold:.4f}')
    ax.set_ylabel('Anomaly score')
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=12)
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(out_path, dpi=320, bbox_inches='tight')
    plt.close(fig)


def plot_curves(setting_name, results_df, y_final, model_scores, prevalence):
    fig, ax = plt.subplots(figsize=(7.2, 5.4), dpi=320)
    for model_name in results_df['model']:
        fpr, tpr, _ = roc_curve(y_final, model_scores[model_name])
        ax.plot(fpr, tpr, linewidth=2, label=model_name)
    ax.plot([0, 1], [0, 1], linestyle='--', linewidth=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curves - {setting_name}')
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, f'roc_curves_{setting_name}.png'), dpi=320, bbox_inches='tight')
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7.2, 5.4), dpi=320)
    for model_name in results_df['model']:
        precision, recall, _ = precision_recall_curve(y_final, model_scores[model_name])
        ax.plot(recall, precision, linewidth=2, label=model_name)
    ax.axhline(prevalence, linestyle='--', linewidth=1, label=f'baseline={prevalence:.4f}')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(f'Precision-Recall Curves - {setting_name}')
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, f'pr_curves_{setting_name}.png'), dpi=320, bbox_inches='tight')
    plt.close(fig)


In [12]:
# وظيفة الخلية: تحميل البيانات الخام من المسار الأصلي وحفظ ملخص الملفات المقروءة.
profiling_raw, attack_test_raw, attack_train_raw, DATA_SOURCE_INFO = resolve_data_source()

save_json({**RUN_CONFIG, 'data_source_info': DATA_SOURCE_INFO}, os.path.join(CONFIG_DIR, 'run_config_initial.json'))

print('Data source mode:', DATA_SOURCE_INFO.get('mode'))
print('Profiling rows:', len(profiling_raw), 'files:', len(DATA_SOURCE_INFO['profiling_files']))
print('Attack-test rows:', len(attack_test_raw), 'files:', len(DATA_SOURCE_INFO['attack_test_files']))
print('Attack-train audit loaded:', attack_train_raw is not None)


بدء نسخ CSV من Drive إلى التخزين المحلي في Colab...
تم تجهيز profiling/CSV: /content/iomt_staged_data/WiFI_and_MQTT/profiling/CSV
تم تجهيز attacks/CSV/test: /content/iomt_staged_data/WiFI_and_MQTT/attacks/CSV/test
تم تجهيز attacks/CSV/train: /content/iomt_staged_data/WiFI_and_MQTT/attacks/CSV/train
Loaded profiling: 10/42 files
Loaded profiling: 20/42 files
Loaded profiling: 30/42 files
Loaded profiling: 40/42 files
Loaded profiling: 42/42 files
Loaded attack_test: 10/21 files
Loaded attack_test: 20/21 files
Loaded attack_test: 21/21 files
Loaded attack_train: 10/51 files
Loaded attack_train: 20/51 files
Loaded attack_train: 30/51 files
Loaded attack_train: 40/51 files
Loaded attack_train: 50/51 files
Loaded attack_train: 51/51 files
Data source mode: staged_local_csv
Profiling rows: 389956 files: 42
Attack-test rows: 1614182 files: 21
Attack-train audit loaded: True


In [13]:
# وظيفة الخلية: اشتقاق خصائص التدريب من profiling فقط ثم تطبيق نفس المخطط على attack-test و attack-train إن وُجد.
FEATURE_SCHEMA = profiling_feature_schema(profiling_raw)
profiling_df, profiling_schema_audit = apply_schema(profiling_raw, FEATURE_SCHEMA)
attack_test_df, attack_test_schema_audit = apply_schema(attack_test_raw, FEATURE_SCHEMA)
attack_train_df, attack_train_schema_audit = apply_schema(attack_train_raw, FEATURE_SCHEMA)

SCHEMA_SUMMARY = {
    'data_source_mode': DATA_SOURCE_INFO.get('mode'),
    'profiling_feature_count': len(FEATURE_SCHEMA),
    'attack_test_missing_columns_from_schema': attack_test_schema_audit.get('missing_from_schema', []),
    'attack_test_extra_numeric_columns_ignored': attack_test_schema_audit.get('extra_numeric_ignored', []),
    'attack_train_missing_columns_from_schema': attack_train_schema_audit.get('missing_from_schema', []),
    'attack_train_extra_numeric_columns_ignored': attack_train_schema_audit.get('extra_numeric_ignored', []),
}

pd.DataFrame({'feature': FEATURE_SCHEMA}).to_csv(os.path.join(CONFIG_DIR, 'feature_schema.csv'), index=False)
save_json(FEATURE_SCHEMA, os.path.join(CONFIG_DIR, 'feature_schema.json'))
save_json(SCHEMA_SUMMARY, os.path.join(CONFIG_DIR, 'feature_schema_summary.json'))
save_json({**RUN_CONFIG, 'data_source_info': DATA_SOURCE_INFO, 'schema_summary': SCHEMA_SUMMARY}, os.path.join(CONFIG_DIR, 'run_config.json'))

print('Profiling-derived feature count:', len(FEATURE_SCHEMA))
print('First features:', FEATURE_SCHEMA[:12])
print('Attack-test missing schema columns:', SCHEMA_SUMMARY['attack_test_missing_columns_from_schema'])
print('Attack-test ignored numeric columns:', SCHEMA_SUMMARY['attack_test_extra_numeric_columns_ignored'])


Profiling-derived feature count: 39
First features: ['ARP', 'AVG', 'DHCP', 'DNS', 'HTTP', 'HTTPS', 'Header_Length', 'IAT', 'ICMP', 'IGMP', 'IPv', 'IRC']
Attack-test missing schema columns: ['Time_To_Live']
Attack-test ignored numeric columns: ['Duration', 'Srate', 'Drate', 'Magnitue', 'Radius', 'Covariance', 'Weight']


In [14]:
# وظيفة الخلية: حساب row hashes والتدقيق الأولي للتكرار والتداخل قبل أي تقسيم أو تدريب.
for df in [profiling_df, attack_test_df]:
    df['__row_hash'] = row_hashes(df, FEATURE_SCHEMA)
if attack_train_df is not None:
    attack_train_df['__row_hash'] = row_hashes(attack_train_df, FEATURE_SCHEMA)

raw_duplicate_rows = [
    duplicate_summary(profiling_df, 'profiling'),
    duplicate_summary(attack_test_df, 'attack_test'),
    duplicate_summary(attack_train_df, 'attack_train'),
]
duplicate_audit_raw_df = pd.DataFrame(raw_duplicate_rows)
duplicate_audit_raw_df.to_csv(os.path.join(AUDITS_DIR, 'duplicate_audit_raw.csv'), index=False)
save_json(raw_duplicate_rows, os.path.join(AUDITS_DIR, 'duplicate_audit_raw.json'))

raw_overlap_rows = [
    {'setting': 'raw', 'pair': 'profiling_vs_attack_test', 'overlap_hashes': overlap_count(profiling_df, attack_test_df), 'available': True},
    {'setting': 'raw', 'pair': 'profiling_vs_attack_train', 'overlap_hashes': overlap_count(profiling_df, attack_train_df), 'available': attack_train_df is not None},
    {'setting': 'raw', 'pair': 'attack_test_vs_attack_train', 'overlap_hashes': overlap_count(attack_test_df, attack_train_df), 'available': attack_train_df is not None},
]
pd.DataFrame(raw_overlap_rows).to_csv(os.path.join(AUDITS_DIR, 'overlap_audit_raw.csv'), index=False)

print('Raw duplicate audit:')
print(duplicate_audit_raw_df.to_string(index=False))
print('\nRaw overlap audit:')
print(pd.DataFrame(raw_overlap_rows).to_string(index=False))


Raw duplicate audit:
        pool    rows  unique_rows  duplicate_rows  duplicate_ratio  available
   profiling  389956       360962           28994         0.074352       True
 attack_test 1614182      1612111            2071         0.001283       True
attack_train 7160831      7155707            5124         0.000716       True

Raw overlap audit:
setting                        pair  overlap_hashes  available
    raw    profiling_vs_attack_test               0       True
    raw   profiling_vs_attack_train               0       True
    raw attack_test_vs_attack_train             197       True


In [15]:
# وظيفة الخلية: تهيئة الحاويات التي ستجمع الجداول والنتائج أثناء تشغيل الإعدادات.
RESULT_ROWS = []
THRESHOLD_ROWS = []
FIT_ROWS = []
SPLIT_ROWS = []
OVERLAP_ROWS = list(raw_overlap_rows)
CONFUSION_JSON = {}
BENIGN_ROWS = []
SCORE_GROUP_ROWS = []
THRESHOLD_SENS_ROWS = []
ALL_PRED_ROWS = []
ALL_BENIGN_PRED_ROWS = []
BENIGN_PREDICTION_PATH_ROWS = []
RUN_SUMMARIES = []


In [16]:
# وظيفة الخلية: تجهيز بيانات إعداد واحد؛ هنا يتم تطبيق dedup وتقسيم normal وحفظ split audit دون تدريب النماذج.
def prepare_setting_data(setting_name, use_dedup):
    profiling_work = profiling_df.drop_duplicates(subset='__row_hash').copy() if use_dedup else profiling_df.copy()
    attack_test_work = attack_test_df.drop_duplicates(subset='__row_hash').copy() if use_dedup else attack_test_df.copy()

    profiling_hashes = set(profiling_work['__row_hash'])
    attack_test_clean = attack_test_work[~attack_test_work['__row_hash'].isin(profiling_hashes)].copy().reset_index(drop=True)

    normal_train, normal_validation, normal_test = split_by_hash(profiling_work, RANDOM_STATE)

    assert_disjoint(normal_train, normal_validation, 'normal_train', 'normal_validation')
    assert_disjoint(normal_train, normal_test, 'normal_train', 'normal_test')
    assert_disjoint(normal_validation, normal_test, 'normal_validation', 'normal_test')
    assert overlap_count(normal_train, attack_test_clean) == 0, 'normal_train overlaps with attack_test_clean'
    assert overlap_count(normal_validation, attack_test_clean) == 0, 'normal_validation overlaps with attack_test_clean'
    assert overlap_count(normal_test, attack_test_clean) == 0, 'normal_test overlaps with attack_test_clean'

    OVERLAP_ROWS.extend([
        {'setting': setting_name, 'pair': 'profiling_vs_attack_test_clean', 'overlap_hashes': overlap_count(profiling_work, attack_test_clean), 'available': True},
        {'setting': setting_name, 'pair': 'normal_train_vs_normal_validation', 'overlap_hashes': overlap_count(normal_train, normal_validation), 'available': True},
        {'setting': setting_name, 'pair': 'normal_train_vs_normal_test', 'overlap_hashes': overlap_count(normal_train, normal_test), 'available': True},
        {'setting': setting_name, 'pair': 'normal_validation_vs_normal_test', 'overlap_hashes': overlap_count(normal_validation, normal_test), 'available': True},
        {'setting': setting_name, 'pair': 'normal_train_vs_attack_test_clean', 'overlap_hashes': overlap_count(normal_train, attack_test_clean), 'available': True},
        {'setting': setting_name, 'pair': 'normal_validation_vs_attack_test_clean', 'overlap_hashes': overlap_count(normal_validation, attack_test_clean), 'available': True},
        {'setting': setting_name, 'pair': 'normal_test_vs_attack_test_clean', 'overlap_hashes': overlap_count(normal_test, attack_test_clean), 'available': True},
    ])

    final_test_df = make_final_test(normal_test, attack_test_clean)
    benign_rows = int(attack_test_clean['__source_file'].eq('Benign_test.pcap.csv').sum())

    SPLIT_ROWS.append({
        'setting': setting_name,
        'profiling_rows_before_dedup': int(len(profiling_df)),
        'profiling_rows_after_dedup': int(len(profiling_work)),
        'attack_test_rows_before_dedup': int(len(attack_test_df)),
        'attack_test_rows_after_dedup': int(len(attack_test_work)),
        'attack_test_clean_rows': int(len(attack_test_clean)),
        'normal_train_rows': int(len(normal_train)),
        'normal_validation_rows': int(len(normal_validation)),
        'normal_test_rows': int(len(normal_test)),
        'final_test_rows': int(len(final_test_df)),
        'positive_class_prevalence': float(final_test_df['final_label'].mean()),
        'benign_test_rows_in_attack_test_clean': benign_rows,
    })

    split_dir = os.path.join(SPLITS_DIR, setting_name)
    os.makedirs(split_dir, exist_ok=True)
    safe_to_parquet(normal_train, os.path.join(split_dir, 'normal_train.parquet'))
    safe_to_parquet(normal_validation, os.path.join(split_dir, 'normal_validation.parquet'))
    safe_to_parquet(normal_test, os.path.join(split_dir, 'normal_test.parquet'))
    safe_to_parquet(attack_test_clean, os.path.join(split_dir, 'attack_test_clean.parquet'))
    safe_to_parquet(final_test_df, os.path.join(split_dir, 'final_test_directory_protocol.parquet'))

    return normal_train, normal_validation, normal_test, attack_test_clean, final_test_df


In [17]:
# وظيفة الخلية: تدريب النماذج لإعداد واحد وتخزين المقاييس والتنبؤات؛ العتبة تؤخذ من normal_validation فقط.
def fit_and_score_models(setting_name, normal_train, normal_validation, final_test_df):
    imputer, scaler, X_train_scaled = fit_preprocessors(normal_train, FEATURE_SCHEMA)
    X_val_scaled = transform_features(normal_validation, FEATURE_SCHEMA, imputer, scaler)
    X_final_scaled = transform_features(final_test_df, FEATURE_SCHEMA, imputer, scaler)
    y_final = final_test_df['final_label'].astype(int).to_numpy()

    assert len(X_train_scaled) == len(normal_train)
    assert len(X_val_scaled) == len(normal_validation)
    assert len(X_final_scaled) == len(final_test_df)

    save_json({'feature_cols': FEATURE_SCHEMA, 'setting': setting_name}, os.path.join(CONFIG_DIR, f'{setting_name}_feature_cols.json'))

    model_scores = {}
    model_preds = {}
    model_thresholds = {}
    validation_scores = {}

    models = build_models(run_ocsvm=RUN_OCSVM_MAIN)
    for model_name, model in models.items():
        fit_start = time.time()
        model.fit(X_train_scaled)
        fit_seconds = time.time() - fit_start
        FIT_ROWS.append({'setting': setting_name, 'model': model_name, 'train_rows_used': int(len(normal_train)), 'fit_time_seconds': float(fit_seconds)})

        val_scores = -model.decision_function(X_val_scaled)
        threshold = float(np.percentile(val_scores, 95))
        final_scores = -model.decision_function(X_final_scaled)
        metrics, preds = evaluate_scores(y_final, final_scores, threshold)

        model_scores[model_name] = final_scores
        model_preds[model_name] = preds
        model_thresholds[model_name] = threshold
        validation_scores[model_name] = val_scores

        row = {'setting': setting_name, 'model': model_name, 'train_rows_used': int(len(normal_train)), 'fit_time_seconds': float(fit_seconds), **metrics}
        RESULT_ROWS.append(row)
        THRESHOLD_ROWS.append({'setting': setting_name, 'model': model_name, 'threshold_percentile': 95, 'threshold': threshold})
        CONFUSION_JSON[f'{setting_name}::{model_name}'] = {'tn': row['tn'], 'fp': row['fp'], 'fn': row['fn'], 'tp': row['tp']}

        pred_df = final_test_df[META_COLS + ['final_label']].copy()
        pred_df['anomaly_score'] = final_scores
        pred_df['prediction'] = preds
        pred_df['model'] = model_name
        pred_df['setting'] = setting_name
        ALL_PRED_ROWS.append(pred_df)

    results_df = pd.DataFrame([r for r in RESULT_ROWS if r['setting'] == setting_name])
    results_df = results_df.sort_values(['balanced_accuracy', 'mcc', 'f1', 'roc_auc'], ascending=False).reset_index(drop=True)
    best_model = results_df.iloc[0]['model']

    return y_final, results_df, best_model, model_scores, model_preds, model_thresholds, validation_scores


In [18]:

# وظيفة الخلية: حساب حساسية Benign_test وتلخيص توزيع الدرجات دون إعادة تدريب.
def add_benign_and_score_summaries(setting_name, final_test_df, model_scores, model_preds, model_thresholds):
    for model_name, final_scores in model_scores.items():
        threshold = model_thresholds[model_name]
        preds = model_preds[model_name]
        pred_df = final_test_df[META_COLS + ['final_label']].copy()
        pred_df['anomaly_score'] = final_scores
        pred_df['prediction'] = preds
        pred_df['model'] = model_name
        pred_df['setting'] = setting_name

        for treatment in ['reported_directory_protocol', 'benign_relabel_as_normal', 'benign_file_excluded']:
            work = pred_df.copy()
            if treatment == 'benign_relabel_as_normal':
                work.loc[work['__source_file'].eq('Benign_test.pcap.csv'), 'final_label'] = 0
            elif treatment == 'benign_file_excluded':
                work = work.loc[~work['__source_file'].eq('Benign_test.pcap.csv')].copy()
            y_t = work['final_label'].astype(int).to_numpy()
            s_t = work['anomaly_score'].to_numpy()
            m_t, p_t = evaluate_scores(y_t, s_t, threshold)
            BENIGN_ROWS.append({'setting': setting_name, 'model': model_name, 'treatment': treatment, **m_t})

            if SAVE_BENIGN_SENSITIVITY_PREDICTIONS:
                # لا نكدس هذه الجداول في الذاكرة؛ نحفظ كل treatment مباشرة في ملف مستقل.
                work['prediction'] = p_t
                work['treatment'] = treatment
                work = optimize_prediction_table(work)
                out_name = f"benign_sensitivity_{safe_token(setting_name)}_{safe_token(model_name)}_{safe_token(treatment)}.parquet"
                out_path = os.path.join(PREDICTIONS_DIR, out_name)
                saved_path = safe_to_parquet(work, out_path)
                BENIGN_PREDICTION_PATH_ROWS.append({
                    'setting': setting_name,
                    'model': model_name,
                    'treatment': treatment,
                    'rows': int(len(work)),
                    'path': saved_path,
                })
            del work
            gc.collect()

        benign_scores = pred_df.loc[pred_df['__source_file'].eq('Benign_test.pcap.csv'), 'anomaly_score'].to_numpy()
        normal_scores = pred_df.loc[pred_df['final_label'].eq(0), 'anomaly_score'].to_numpy()
        other_attack_scores = pred_df.loc[(pred_df['final_label'].eq(1)) & (~pred_df['__source_file'].eq('Benign_test.pcap.csv')), 'anomaly_score'].to_numpy()
        for group_name, scores in [('normal_test', normal_scores), ('Benign_test.pcap.csv', benign_scores), ('other_attack_test_rows', other_attack_scores)]:
            row = {'setting': setting_name, 'model': model_name, 'group': group_name}
            row.update(score_summary(scores, threshold))
            SCORE_GROUP_ROWS.append(row)

        del pred_df
        gc.collect()


In [19]:
# وظيفة الخلية: إنتاج تحليل حساسية العتبة للنموذج الأفضل دون تدريب إضافي.
def add_threshold_sensitivity(setting_name, best_model, y_final, validation_scores, model_scores):
    if not RUN_THRESHOLD_SENSITIVITY:
        return
    for pct in THRESHOLD_PERCENTILES:
        threshold = float(np.percentile(validation_scores[best_model], pct))
        metrics, _ = evaluate_scores(y_final, model_scores[best_model], threshold)
        THRESHOLD_SENS_ROWS.append({'setting': setting_name, 'model': best_model, 'threshold_percentile': pct, **metrics})


In [20]:
# وظيفة الخلية: رسم مخرجات الإعداد الواحد بعد التدريب، مع اختيار أفضل نموذج حسب المقاييس التشغيلية.
def plot_setting_outputs(setting_name, y_final, final_test_df, results_df, best_model, model_scores, model_preds, model_thresholds):
    prevalence = float(final_test_df['final_label'].mean())
    plot_curves(setting_name, results_df, y_final, model_scores, prevalence)

    cm = confusion_matrix(y_final, model_preds[best_model], labels=[0, 1])
    plot_confusion_matrix(
        cm,
        f'Confusion Matrix - {setting_name} - {best_model}',
        os.path.join(FIGURES_DIR, f'confusion_matrix_best_{setting_name}.png'),
    )

    best_scores = model_scores[best_model]
    normal_scores = best_scores[final_test_df['final_label'].to_numpy() == 0]
    benign_scores = best_scores[final_test_df['__source_file'].eq('Benign_test.pcap.csv').to_numpy()]
    other_attack_scores = best_scores[(final_test_df['final_label'].to_numpy() == 1) & (~final_test_df['__source_file'].eq('Benign_test.pcap.csv').to_numpy())]
    plot_score_boxplot(
        [('normal_test', normal_scores), ('Benign_test.pcap.csv', benign_scores), ('other_attack_test_rows', other_attack_scores)],
        model_thresholds[best_model],
        f'Score Distribution - {setting_name} - {best_model}',
        os.path.join(FIGURES_DIR, f'score_distribution_groups_{setting_name}.png'),
    )


In [21]:
# وظيفة الخلية: تشغيل إعداد واحد كاملاً من التقسيم حتى الرسوم، ثم إعادة ملخص يسمح بتوثيق البروتوكول.
def run_setting(setting_name, use_dedup):
    normal_train, normal_validation, normal_test, attack_test_clean, final_test_df = prepare_setting_data(setting_name, use_dedup)
    y_final, results_df, best_model, model_scores, model_preds, model_thresholds, validation_scores = fit_and_score_models(
        setting_name, normal_train, normal_validation, final_test_df
    )
    add_benign_and_score_summaries(setting_name, final_test_df, model_scores, model_preds, model_thresholds)
    add_threshold_sensitivity(setting_name, best_model, y_final, validation_scores, model_scores)
    plot_setting_outputs(setting_name, y_final, final_test_df, results_df, best_model, model_scores, model_preds, model_thresholds)

    del normal_train, normal_validation, normal_test, attack_test_clean, final_test_df
    gc.collect()

    return {
        'setting': setting_name,
        'best_model': best_model,
        'feature_count': len(FEATURE_SCHEMA),
        'independent_normal_test_implemented': True,
        'threshold_uses_only_normal_validation': True,
        'final_test_uses_only_normal_test_plus_attack_test_clean': True,
    }


In [22]:
# وظيفة الخلية: تشغيل الإعدادات الرئيسية المحددة في الأعلى فقط، وفق إعدادات التشغيل النهائي؛ repeated seeds في خلية مستقلة لاحقة.
if RUN_MAIN:
    if RUN_DEDUP_ON:
        RUN_SUMMARIES.append(run_setting('dedup_on', True))
    if RUN_DEDUP_OFF:
        RUN_SUMMARIES.append(run_setting('dedup_off', False))
else:
    print('RUN_MAIN=False؛ لم يتم تدريب النماذج الرئيسية.')

print('Completed settings:', RUN_SUMMARIES)


/tmp/ipykernel_36602/2365907949.py:33: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_df = pd.concat([normal_part, anomaly_side], ignore_index=True)
/tmp/ipykernel_36602/1502752364.py:33: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(values, labels=labels, showfliers=False)
/tmp/ipykernel_36602/2365907949.py:33: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
 

Completed settings: [{'setting': 'dedup_on', 'best_model': 'ocsvm', 'feature_count': 39, 'independent_normal_test_implemented': True, 'threshold_uses_only_normal_validation': True, 'final_test_uses_only_normal_test_plus_attack_test_clean': True}, {'setting': 'dedup_off', 'best_model': 'ocsvm', 'feature_count': 39, 'independent_normal_test_implemented': True, 'threshold_uses_only_normal_validation': True, 'final_test_uses_only_normal_test_plus_attack_test_clean': True}]


In [23]:
# وظيفة الخلية: تحليل اختياري للبذور المتعددة؛ مفعّل في نسخة التشغيل النهائي لإنتاج جدول الثبات عبر البذور.
REPEATED_METRIC_ROWS = []
REPEATED_SUMMARY_ROWS = []

if RUN_REPEATED_SEEDS:
    for setting_name, use_dedup in [('dedup_on', True), ('dedup_off', False)]:
        if setting_name == 'dedup_on' and not RUN_DEDUP_ON:
            continue
        if setting_name == 'dedup_off' and not RUN_DEDUP_OFF:
            continue
        profiling_work = profiling_df.drop_duplicates(subset='__row_hash').copy() if use_dedup else profiling_df.copy()
        attack_test_work = attack_test_df.drop_duplicates(subset='__row_hash').copy() if use_dedup else attack_test_df.copy()
        profiling_hashes = set(profiling_work['__row_hash'])
        attack_test_clean = attack_test_work[~attack_test_work['__row_hash'].isin(profiling_hashes)].copy().reset_index(drop=True)

        for seed in REPEATED_SEEDS:
            normal_train, normal_validation, normal_test = split_by_hash(profiling_work, seed)
            final_test_df = make_final_test(normal_test, attack_test_clean)
            imputer, scaler, X_train_scaled = fit_preprocessors(normal_train, FEATURE_SCHEMA)
            X_val_scaled = transform_features(normal_validation, FEATURE_SCHEMA, imputer, scaler)
            X_final_scaled = transform_features(final_test_df, FEATURE_SCHEMA, imputer, scaler)
            y_final = final_test_df['final_label'].astype(int).to_numpy()

            for model_name, model in build_models(run_ocsvm=RUN_OCSVM_REPEATED, seed=seed).items():
                fit_start = time.time()
                model.fit(X_train_scaled)
                fit_seconds = time.time() - fit_start
                val_scores = -model.decision_function(X_val_scaled)
                threshold = float(np.percentile(val_scores, 95))
                final_scores = -model.decision_function(X_final_scaled)
                metrics, _ = evaluate_scores(y_final, final_scores, threshold)
                REPEATED_METRIC_ROWS.append({'setting': setting_name, 'seed': seed, 'model': model_name, 'fit_time_seconds': fit_seconds, **metrics})

            del normal_train, normal_validation, normal_test, final_test_df, X_train_scaled, X_val_scaled, X_final_scaled
            gc.collect()

    if REPEATED_METRIC_ROWS:
        rep_df = pd.DataFrame(REPEATED_METRIC_ROWS)
        rep_df.to_csv(os.path.join(METRICS_DIR, 'repeated_seed_metrics.csv'), index=False)
        numeric_cols = ['roc_auc', 'pr_auc', 'f1', 'mcc', 'specificity', 'fpr', 'balanced_accuracy']
        agg = rep_df.groupby(['setting', 'model'])[numeric_cols].agg(['mean', 'std', 'min', 'max', 'count']).reset_index()
        agg.to_csv(os.path.join(METRICS_DIR, 'repeated_seed_summary.csv'), index=False)
        print(agg.to_string(index=False))
else:
    print('Repeated-seed analysis skipped. فعّل RUN_REPEATED_SEEDS إذا احتجته للنسخة النهائية.')


/tmp/ipykernel_36602/2365907949.py:33: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_df = pd.concat([normal_part, anomaly_side], ignore_index=True)
/tmp/ipykernel_36602/2365907949.py:33: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_df = pd.concat([normal_part, anomaly_side], ignore_index=True)
/tmp/ipykernel_36602/2365907949.py:33: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty

  setting            model  roc_auc                                    pr_auc                                            f1                                       mcc                                  specificity                                       fpr                                  balanced_accuracy                                 
                               mean      std      min      max count     mean          std      min      max count     mean      std      min      max count     mean      std      min      max count        mean      std      min      max count     mean      std      min      max count              mean      std      min      max count
dedup_off isolation_forest 0.945009 0.006747 0.934807 0.952714     5 0.995247 7.534816e-04 0.994123 0.996215     5 0.763292 0.114927 0.676262 0.889399     5 0.267083 0.097499 0.193757 0.374253     5    0.950839 0.000987 0.949322 0.951923     5 0.049161 0.000987 0.048077 0.050678     5          0.790589 0.078479 0.730860 0.87

In [24]:
# وظيفة الخلية: حفظ كل الجداول الأساسية والتنبؤات بعد اكتمال التدريب الرئيسي.
metrics_main_df = pd.DataFrame(RESULT_ROWS)
if len(metrics_main_df):
    metrics_main_df = metrics_main_df.sort_values(['setting', 'balanced_accuracy', 'mcc', 'f1', 'roc_auc'], ascending=[True, False, False, False, False]).reset_index(drop=True)

split_summary_df = pd.DataFrame(SPLIT_ROWS)
overlap_audit_df = pd.DataFrame(OVERLAP_ROWS)
threshold_summary_df = pd.DataFrame(THRESHOLD_ROWS)
fit_times_df = pd.DataFrame(FIT_ROWS)
benign_metrics_df = pd.DataFrame(BENIGN_ROWS)
score_group_summary_df = pd.DataFrame(SCORE_GROUP_ROWS)
threshold_sensitivity_df = pd.DataFrame(THRESHOLD_SENS_ROWS)

metrics_main_df.to_csv(os.path.join(METRICS_DIR, 'metrics_main.csv'), index=False)
save_json(metrics_main_df.to_dict(orient='records'), os.path.join(METRICS_DIR, 'metrics_main.json'))
threshold_summary_df.to_csv(os.path.join(METRICS_DIR, 'threshold_summary.csv'), index=False)
fit_times_df.to_csv(os.path.join(METRICS_DIR, 'fit_times.csv'), index=False)
benign_metrics_df.to_csv(os.path.join(METRICS_DIR, 'benign_sensitivity_metrics.csv'), index=False)
score_group_summary_df.to_csv(os.path.join(METRICS_DIR, 'score_group_summary.csv'), index=False)
threshold_sensitivity_df.to_csv(os.path.join(METRICS_DIR, 'threshold_sensitivity.csv'), index=False)

duplicate_audit_raw_df.to_csv(os.path.join(AUDITS_DIR, 'duplicate_audit_raw.csv'), index=False)
overlap_audit_df.to_csv(os.path.join(AUDITS_DIR, 'overlap_audit.csv'), index=False)
split_summary_df.to_csv(os.path.join(SUMMARIES_DIR, 'split_summary.csv'), index=False)
save_json(CONFUSION_JSON, os.path.join(METRICS_DIR, 'confusion_matrices.json'))

pred_long_df = pd.concat(ALL_PRED_ROWS, ignore_index=True) if ALL_PRED_ROWS else pd.DataFrame()
if len(pred_long_df):
    pred_long_df = optimize_prediction_table(pred_long_df)
    safe_to_parquet(pred_long_df, os.path.join(PREDICTIONS_DIR, 'test_predictions.parquet'))
    if SAVE_LARGE_PREDICTION_CSV:
        pred_long_df.to_csv(os.path.join(PREDICTIONS_DIR, 'test_predictions.csv'), index=False)

if SAVE_BENIGN_SENSITIVITY_PREDICTIONS:
    benign_prediction_files_df = pd.DataFrame(BENIGN_PREDICTION_PATH_ROWS)
    benign_prediction_files_df.to_csv(os.path.join(PREDICTIONS_DIR, 'benign_sensitivity_prediction_files.csv'), index=False)
else:
    print('Benign sensitivity full predictions were not saved; metrics are saved and the analysis is reproducible from test_predictions.')

print('Saved outputs under:', EXPERIMENT_ROOT)
print('Main metrics:')
print(metrics_main_df.to_string(index=False))


Saved outputs under: /content/iomt_publication_runs/publication_full_20260506_013718
Main metrics:
  setting            model  train_rows_used  fit_time_seconds  roc_auc   pr_auc    tn   fp     fn      tp  threshold  accuracy  precision   recall       f1      mcc  specificity      fpr  balanced_accuracy  pr_auc_baseline
dedup_off            ocsvm           234082        692.919478 1.000000 1.000000 73731 3900      0 1614182   0.007314  0.997695   0.997590 1.000000 0.998793 0.973382     0.949762 0.050238           0.974881         0.954114
dedup_off              lof           234082         93.196016 0.993215 0.998645 73897 3734  12244 1601938   2.113561  0.990556   0.997674 0.992415 0.995038 0.898830     0.951901 0.048099           0.972158         0.954114
dedup_off isolation_forest           234082          0.764824 0.928565 0.993335 73736 3895 805646  808536   0.030413  0.521495   0.995206 0.500895 0.666391 0.188765     0.949827 0.050173           0.725361         0.954114
 dedup_on

In [25]:
# وظيفة الخلية: تصنيف أسماء ملفات الهجوم إلى عائلات؛ يستخدم فقط لجداول النشر ولا يغير التدريب.
def infer_attack_family_from_file(filename):
    name = str(filename).replace('.pcap.csv', '')
    if name == 'Benign_test':
        return 'Benign test file'
    if name.startswith('ARP_Spoofing'):
        return 'ARP spoofing'
    if name.startswith('Recon-'):
        return 'Reconnaissance'
    if name.startswith('MQTT-'):
        if 'Malformed' in name:
            return 'MQTT malformed data'
        if 'Connect_Flood' in name:
            return 'MQTT connect flood'
        if 'Publish_Flood' in name:
            return 'MQTT publish flood'
        return 'MQTT other'
    if name.startswith('TCP_IP-DDoS-'):
        return 'TCP/IP DDoS'
    if name.startswith('TCP_IP-DoS-'):
        return 'TCP/IP DoS'
    return 'Profiling normal'


def summarize_predictions(group, threshold=None):
    y = group['final_label'].astype(int).to_numpy()
    p = group['prediction'].astype(int).to_numpy()
    s = group['anomaly_score'].astype(float).to_numpy()
    tn = int(((y == 0) & (p == 0)).sum())
    fp = int(((y == 0) & (p == 1)).sum())
    fn = int(((y == 1) & (p == 0)).sum())
    tp = int(((y == 1) & (p == 1)).sum())
    out = confusion_to_metrics(tn, fp, fn, tp)
    median_score = float(np.median(s)) if len(s) else np.nan
    out.update({
        'rows': int(len(group)), 'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
        'median_score': median_score,
        'q1_score': float(np.quantile(s, 0.25)) if len(s) else np.nan,
        'q3_score': float(np.quantile(s, 0.75)) if len(s) else np.nan,
        'min_score': float(np.min(s)) if len(s) else np.nan,
        'max_score': float(np.max(s)) if len(s) else np.nan,
        'pct_predicted_anomaly': float(p.mean() * 100.0) if len(p) else np.nan,
        'threshold': float(threshold) if threshold is not None else np.nan,
        'median_score_minus_threshold': float(median_score - threshold) if threshold is not None and len(s) else np.nan,
    })
    return out


In [26]:
# وظيفة الخلية: إنتاج جداول النشر المطلوبة: attack-only، per-file، per-family، وfalse positives للملفات الطبيعية.
if 'pred_long_df' not in globals() or pred_long_df is None or len(pred_long_df) == 0:
    parquet_path = os.path.join(PREDICTIONS_DIR, 'test_predictions.parquet')
    pickle_path = os.path.join(PREDICTIONS_DIR, 'test_predictions.pkl')
    csv_path = os.path.join(PREDICTIONS_DIR, 'test_predictions.csv')
    if os.path.exists(parquet_path):
        pred_long_df = pd.read_parquet(parquet_path)
    elif os.path.exists(pickle_path):
        pred_long_df = pd.read_pickle(pickle_path)
    elif os.path.exists(csv_path):
        pred_long_df = pd.read_csv(csv_path)
    else:
        raise FileNotFoundError('No saved prediction table found in predictions directory.')

threshold_lookup = threshold_summary_df.set_index(['setting', 'model'])['threshold'].to_dict() if len(threshold_summary_df) else {}

attack_only_metrics_df = benign_metrics_df[benign_metrics_df['treatment'].eq('benign_file_excluded')].copy()
attack_only_metrics_df.to_csv(os.path.join(PUBLICATION_DIR, 'attack_only_metrics_excluding_benign.csv'), index=False)

work_all = pred_long_df.copy()
work_all['attack_family'] = work_all['__source_file'].astype(str).map(infer_attack_family_from_file)

attack_file_rows = []
attack_family_rows = []
normal_file_rows = []

for (setting, model), sub in work_all.groupby(['setting', 'model'], sort=True, observed=False):
    threshold = threshold_lookup.get((setting, model), np.nan)

    attack_only = sub[(sub['final_label'].astype(int).eq(1)) & (~sub['__source_file'].astype(str).eq('Benign_test.pcap.csv'))].copy()
    for source_file, g in attack_only.groupby('__source_file', sort=True, observed=False):
        row = {'setting': setting, 'model': model, 'source_file': source_file, 'attack_family': infer_attack_family_from_file(source_file)}
        row.update(summarize_predictions(g, threshold))
        attack_file_rows.append(row)
    for attack_family, g in attack_only.groupby('attack_family', sort=True, observed=False):
        row = {'setting': setting, 'model': model, 'attack_family': attack_family}
        row.update(summarize_predictions(g, threshold))
        attack_family_rows.append(row)

    normal_only = sub[sub['final_label'].astype(int).eq(0)].copy()
    for source_file, g in normal_only.groupby('__source_file', sort=True, observed=False):
        row = {'setting': setting, 'model': model, 'source_file': source_file}
        row.update(summarize_predictions(g, threshold))
        normal_file_rows.append(row)

per_attack_file_df = pd.DataFrame(attack_file_rows)
per_attack_family_df = pd.DataFrame(attack_family_rows)
normal_file_fp_df = pd.DataFrame(normal_file_rows)

per_attack_file_df.to_csv(os.path.join(PUBLICATION_DIR, 'per_attack_file_metrics.csv'), index=False)
per_attack_family_df.to_csv(os.path.join(PUBLICATION_DIR, 'per_attack_family_metrics.csv'), index=False)
normal_file_fp_df.to_csv(os.path.join(PUBLICATION_DIR, 'normal_file_false_positive_metrics.csv'), index=False)

main_family_table = per_attack_family_df[
    per_attack_family_df['setting'].eq('dedup_on') & per_attack_family_df['model'].eq('ocsvm')
].copy()
if len(main_family_table):
    keep_cols = ['attack_family', 'rows', 'fn', 'tp', 'recall', 'median_score', 'median_score_minus_threshold']
    main_family_table = main_family_table[keep_cols]
    main_family_table.to_csv(os.path.join(PUBLICATION_DIR, 'manuscript_ocsvm_dedup_on_attack_family_table.csv'), index=False)

print('Publication tables saved:', PUBLICATION_DIR)
print(main_family_table.to_string(index=False) if len(main_family_table) else 'No OCSVM dedup_on family table; check RUN_OCSVM_MAIN.')


Publication tables saved: /content/iomt_publication_runs/publication_full_20260506_013718/publication_checks
      attack_family    rows  fn      tp  recall  median_score  median_score_minus_threshold
       ARP spoofing    1744   0    1744     1.0    356.489215                    356.475659
 MQTT connect flood   45047   0   45047     1.0    356.489215                    356.475659
MQTT malformed data    1747   0    1747     1.0    356.489215                    356.475659
 MQTT publish flood   16921   0   16921     1.0    356.489215                    356.475659
     Reconnaissance   25607   0   25607     1.0    356.489215                    356.475659
        TCP/IP DDoS 1066762   0 1066762     1.0    356.489215                    356.475659
         TCP/IP DoS  416676   0  416676     1.0    356.489215                    356.475659


In [27]:
# وظيفة الخلية: إنشاء manifest بقيم SHA-256 لكل مخرجات التجربة لإثبات القابلية لإعادة الإنتاج.
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

manifest_rows = []
for root, _, files in os.walk(EXPERIMENT_ROOT):
    for name in files:
        if name.endswith(('.csv', '.json', '.png', '.parquet', '.ipynb')):
            full = os.path.join(root, name)
            rel = os.path.relpath(full, EXPERIMENT_ROOT)
            manifest_rows.append({'relative_path': rel, 'size_bytes': os.path.getsize(full), 'sha256': sha256_file(full)})

manifest_df = pd.DataFrame(manifest_rows).sort_values('relative_path')
manifest_df.to_csv(os.path.join(PUBLICATION_DIR, 'reproducibility_manifest_sha256.csv'), index=False)

if_note = {
    'isolation_forest_setting': 'contamination=auto',
    'reported_threshold_policy': '95th percentile of normal_validation anomaly scores',
    'interpretation': 'The reported decision threshold is external and validation-only; contamination is retained as a reproducible model baseline setting and should not be interpreted as attack-prevalence tuning.',
}
save_json(if_note, os.path.join(PUBLICATION_DIR, 'isolation_forest_threshold_policy_note.json'))

print('Manifest rows:', len(manifest_df))
print('Manifest path:', os.path.join(PUBLICATION_DIR, 'reproducibility_manifest_sha256.csv'))


Manifest rows: 66
Manifest path: /content/iomt_publication_runs/publication_full_20260506_013718/publication_checks/reproducibility_manifest_sha256.csv


In [28]:

# وظيفة إضافية: ضغط مجلد النتائج المحلي ونسخه إلى Google Drive بعد اكتمال التشغيل.
def copy_final_results_zip_to_drive():
    if not WRITE_RESULTS_ZIP_TO_DRIVE:
        return None
    if not os.path.isdir(EXPERIMENT_ROOT):
        print('تحذير: لا يوجد EXPERIMENT_ROOT لضغطه:', EXPERIMENT_ROOT)
        return None

    local_zip_base = os.path.join('/content', os.path.basename(EXPERIMENT_ROOT))
    local_zip_path = shutil.make_archive(local_zip_base, 'zip', EXPERIMENT_ROOT)
    print('تم إنشاء أرشيف محلي:', local_zip_path)

    try:
        os.makedirs(DRIVE_EXPERIMENT_PARENT, exist_ok=True)
        drive_zip_path = os.path.join(DRIVE_EXPERIMENT_PARENT, os.path.basename(local_zip_path))
        shutil.copy2(local_zip_path, drive_zip_path)
        print('تم نسخ أرشيف النتائج إلى Drive:', drive_zip_path)
        return drive_zip_path
    except Exception as e:
        print('تحذير: فشل نسخ الأرشيف إلى Drive. الأرشيف المحلي موجود ويمكن تنزيله من Colab.')
        print('سبب الفشل:', type(e).__name__, e)
        return None

FINAL_RESULTS_ZIP_ON_DRIVE = copy_final_results_zip_to_drive()

# وظيفة الخلية: طباعة تقرير نهائي مختصر يوضح مكان النتائج والجداول التي تحتاج إرسالها لتحديث الورقة.
print('\n' + '=' * 90)
print('Experiment root / مسار التجربة:')
print(EXPERIMENT_ROOT)
print('=' * 90)

print('\nFeature count / عدد الخصائص:', len(FEATURE_SCHEMA))
print('\nIndependent normal_test implemented:', bool(RUN_SUMMARIES and all(x['independent_normal_test_implemented'] for x in RUN_SUMMARIES)))
print('Threshold calibration uses only normal_validation:', bool(RUN_SUMMARIES and all(x['threshold_uses_only_normal_validation'] for x in RUN_SUMMARIES)))
print('Final testing uses normal_test + attack_test_clean:', bool(RUN_SUMMARIES and all(x['final_test_uses_only_normal_test_plus_attack_test_clean'] for x in RUN_SUMMARIES)))

print('\nRaw duplicate audit:')
print(duplicate_audit_raw_df.to_string(index=False))
print('\nSplit summary:')
print(split_summary_df.to_string(index=False))
print('\nMain metrics:')
print(metrics_main_df.to_string(index=False))
print('\nBenign sensitivity:')
print(benign_metrics_df.to_string(index=False))
print('\nThreshold sensitivity:')
print(threshold_sensitivity_df.to_string(index=False))

needed_files = [
    os.path.join(METRICS_DIR, 'metrics_main.csv'),
    os.path.join(METRICS_DIR, 'benign_sensitivity_metrics.csv'),
    os.path.join(METRICS_DIR, 'threshold_sensitivity.csv'),
    os.path.join(PUBLICATION_DIR, 'attack_only_metrics_excluding_benign.csv'),
    os.path.join(PUBLICATION_DIR, 'per_attack_family_metrics.csv'),
    os.path.join(PUBLICATION_DIR, 'per_attack_file_metrics.csv'),
    os.path.join(PUBLICATION_DIR, 'normal_file_false_positive_metrics.csv'),
    os.path.join(PUBLICATION_DIR, 'reproducibility_manifest_sha256.csv'),
]
print('\nSend these files for paper finalization / أرسل هذه الملفات لتحديث الورقة:')
for p in needed_files:
    print('-', p)


تم إنشاء أرشيف محلي: /content/publication_full_20260506_013718.zip
تم نسخ أرشيف النتائج إلى Drive: /content/drive/MyDrive/IoMT_Project/Experiments/CICIoMT2024_NormalOnly_DuplicateAware_publication_full/publication_full_20260506_013718.zip

Experiment root / مسار التجربة:
/content/iomt_publication_runs/publication_full_20260506_013718

Feature count / عدد الخصائص: 39

Independent normal_test implemented: True
Threshold calibration uses only normal_validation: True
Final testing uses normal_test + attack_test_clean: True

Raw duplicate audit:
        pool    rows  unique_rows  duplicate_rows  duplicate_ratio  available
   profiling  389956       360962           28994         0.074352       True
 attack_test 1614182      1612111            2071         0.001283       True
attack_train 7160831      7155707            5124         0.000716       True

Split summary:
  setting  profiling_rows_before_dedup  profiling_rows_after_dedup  attack_test_rows_before_dedup  attack_test_rows_after_ded